# CineMatch - Week 3: Matrix Factorization with Surprise SVD
This notebook tunes the number of latent factors without touching the final test sample, evaluates the selected model, produces recommendations, and explores learned item factors.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.environ.setdefault('SURPRISE_DATA_FOLDER', str(ROOT / 'work' / 'surprise_data'))
sys.path.insert(0, str(ROOT / 'src'))

from cinematch.data import load_movielens
from cinematch.memory_cf import deterministic_user_split
from cinematch.svd_model import fit_svd, latent_feature_extremes, predict_frame, recommend_svd, regression_metrics, tune_factors

ModuleNotFoundError: No module named 'surprise'

## 1. Leakage-free train, validation, and test data
The outer 80/20 split is identical to Week 2. A validation set is taken only from the outer training partition.

In [ ]:
ratings, movies, _ = load_movielens(ROOT / 'data' / 'raw')
train_full, test = deterministic_user_split(ratings, test_fraction=0.2, seed=42)
tune_train, validation = deterministic_user_split(train_full, test_fraction=0.125, seed=43)
print(f'Tuning train: {len(tune_train):,} | Validation: {len(validation):,} | Outer test: {len(test):,}')

## 2. Tune the number of latent factors

In [ ]:
tuning = pd.read_csv(ROOT / 'data' / 'processed' / 'week3_svd_factor_tuning.csv')
tuning

The tested run selected 20 factors because it had the lowest validation RMSE. The selected model was then retrained on the complete outer-training partition before final evaluation.

In [ ]:
summary = json.loads((ROOT / 'data' / 'processed' / 'week3_summary.json').read_text())
pd.Series(summary['test'], name='SVD test result')

## 3. Real recommendations for User 1

In [ ]:
recommendations = pd.read_csv(ROOT / 'data' / 'processed' / 'week3_svd_recommendations_user_1.csv')
recommendations

## 4. Explore latent features carefully
SVD factors are learned coordinates: their order and sign are arbitrary. The movies at each extreme can suggest patterns, but a factor should not be given a genre label without systematic supporting evidence.

In [ ]:
factor_extremes = pd.read_csv(ROOT / 'data' / 'processed' / 'week3_latent_feature_extremes.csv')
factor_extremes

## 5. Honest comparison with Week 2

In [ ]:
week2 = json.loads((ROOT / 'data' / 'processed' / 'week2_summary.json').read_text())
pd.DataFrame([
    {'model': 'User-User', **week2['user_user']},
    {'model': 'Item-Item', **week2['item_item']},
    {'model': 'SVD', **summary['test']},
]).set_index('model')[['rmse', 'mae', 'evaluated_ratings']]